# Creazione dataset noisy

Questo notebook genera i dataset noisy per invoice matching a partire da `data/raw/invoices_dataset.json`.

Include errori artificiali didattici come OCR, date/importi alterati, valori mancanti, swap buyer/seller e descrizioni prodotto riordinate o sostituite.

Output principali:
- `data/processed/invoices_noisy_matching.csv`
- `data/processed/invoices_noisy_matching.jsonl`
- `data/processed/invoices_ground_truth_corpus.jsonl`
- `data/processed/invoices_noisy_evaluation.csv`
- `data/processed/invoices_noisy_evaluation.jsonl`

## 1. Funzioni

La logica resta esplicita e in Python semplice, cosi' il notebook e' leggibile e modificabile durante gli esperimenti.

In [ ]:
"""Create a didactic invoice-matching dataset with synthetic OCR/operator errors.

The raw dataset is kept unchanged. This script extracts the ground-truth invoice
JSON from the original conversation format, creates a noisy copy, and exports a
flat CSV plus a JSONL file for later matching/search experiments.
"""

from __future__ import annotations

import csv
import json
import random
import re
from copy import deepcopy
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any


DEFAULT_INPUT = Path("data/raw/invoices_dataset.json")
DEFAULT_CSV_OUTPUT = Path("data/processed/invoices_noisy_matching.csv")
DEFAULT_JSONL_OUTPUT = Path("data/processed/invoices_noisy_matching.jsonl")
DEFAULT_CORPUS_OUTPUT = Path("data/processed/invoices_ground_truth_corpus.jsonl")
DEFAULT_EVAL_CSV_OUTPUT = Path("data/processed/invoices_noisy_evaluation.csv")
DEFAULT_EVAL_JSONL_OUTPUT = Path("data/processed/invoices_noisy_evaluation.jsonl")


OCR_CONFUSIONS = {
    "0": "O",
    "O": "0",
    "o": "0",
    "1": "I",
    "I": "1",
    "l": "1",
    "5": "S",
    "S": "5",
    "8": "B",
    "B": "8",
    "2": "Z",
    "Z": "2",
}

LEGAL_SUFFIX_PATTERNS = [
    r"\bLLC\b",
    r"\bLTD\b",
    r"\bINC\.?\b",
    r"\bCORP\.?\b",
    r"\bS\.?R\.?L\.?\b",
    r"\bS\.?P\.?A\.?\b",
    r"\bPLC\b",
]


@dataclass
class NoiseTrace:
    level: str
    errors: list[str] = field(default_factory=list)

    def add(self, error_type: str) -> None:
        self.errors.append(error_type)



def load_ground_truth(path: Path) -> list[dict[str, Any]]:
    with path.open("r", encoding="utf-8") as file:
        rows = json.load(file)

    parsed_rows: list[dict[str, Any]] = []
    for idx, row in enumerate(rows):
        try:
            invoice_json = row["conversations"][1]["value"]
            ground_truth = json.loads(invoice_json)
        except (KeyError, IndexError, TypeError, json.JSONDecodeError) as exc:
            raise ValueError(f"Could not parse invoice JSON at row {idx}") from exc

        parsed_rows.append(
            {
                "record_id": idx + 1,
                "source_image": row.get("image"),
                "ground_truth": ground_truth,
            }
        )
    return parsed_rows


def choose_noise_level(rng: random.Random) -> str:
    return rng.choices(
        population=["low", "medium", "high"],
        weights=[0.70, 0.20, 0.10],
        k=1,
    )[0]


def operations_for_level(level: str, rng: random.Random) -> int:
    if level == "low":
        return rng.randint(1, 3)
    if level == "medium":
        return rng.randint(4, 7)
    return rng.randint(8, 12)


def structural_noise_probabilities(level: str) -> dict[str, float]:
    """Return probabilities for harder, cross-field synthetic errors."""
    if level == "low":
        return {
            "buyer_seller_swap": 0.02,
            "product_description_shuffle": 0.08,
            "product_description_replace": 0.03,
        }
    if level == "medium":
        return {
            "buyer_seller_swap": 0.18,
            "product_description_shuffle": 0.15,
            "product_description_replace": 0.25,
        }
    return {
        "buyer_seller_swap": 0.35,
        "product_description_shuffle": 0.25,
        "product_description_replace": 0.50,
    }


def maybe(value: Any) -> bool:
    return value is not None and value != ""


def random_char_position(text: str, rng: random.Random) -> int | None:
    positions = [idx for idx, char in enumerate(text) if char.strip()]
    if not positions:
        return None
    return rng.choice(positions)


def apply_ocr_confusion(text: str, rng: random.Random) -> str | None:
    candidates = [idx for idx, char in enumerate(text) if char in OCR_CONFUSIONS]
    if not candidates:
        return None
    idx = rng.choice(candidates)
    return text[:idx] + OCR_CONFUSIONS[text[idx]] + text[idx + 1 :]


def drop_random_char(text: str, rng: random.Random) -> str | None:
    if len(text.strip()) <= 3:
        return None
    idx = random_char_position(text, rng)
    if idx is None:
        return None
    return text[:idx] + text[idx + 1 :]


def insert_random_space(text: str, rng: random.Random) -> str | None:
    if len(text) <= 4:
        return None
    idx = rng.randint(1, len(text) - 1)
    return text[:idx] + " " + text[idx:]


def swap_adjacent_chars(text: str, rng: random.Random) -> str | None:
    positions = [
        idx
        for idx in range(len(text) - 1)
        if text[idx].strip() and text[idx + 1].strip()
    ]
    if not positions:
        return None
    idx = rng.choice(positions)
    return text[:idx] + text[idx + 1] + text[idx] + text[idx + 2 :]


def change_case(text: str, rng: random.Random) -> str:
    if rng.random() < 0.5:
        return text.upper()
    return text.lower()


def abbreviate_company(text: str, rng: random.Random) -> str | None:
    replacements = [
        ("Company", "Co."),
        ("Corporation", "Corp."),
        ("Limited", "Ltd."),
        ("International", "Intl."),
        ("and", "&"),
    ]
    candidates = [(src, dst) for src, dst in replacements if src in text]
    if candidates:
        src, dst = rng.choice(candidates)
        return text.replace(src, dst, 1)

    stripped = text
    for pattern in LEGAL_SUFFIX_PATTERNS:
        stripped = re.sub(pattern, "", stripped, flags=re.IGNORECASE).strip(" ,.-")
    if stripped and stripped != text:
        return stripped
    return None


def corrupt_text(value: Any, rng: random.Random) -> tuple[Any, str] | None:
    if not maybe(value):
        return None

    text = str(value)
    operations = [
        ("ocr_char_confusion", apply_ocr_confusion),
        ("missing_character", drop_random_char),
        ("extra_space", insert_random_space),
        ("transposed_characters", swap_adjacent_chars),
        ("case_variation", lambda text, rng: change_case(text, rng)),
    ]
    rng.shuffle(operations)
    for error_type, operation in operations:
        corrupted = operation(text, rng)
        if corrupted and corrupted != text:
            return corrupted, error_type
    return None


def corrupt_company(value: Any, rng: random.Random) -> tuple[Any, str] | None:
    if not maybe(value):
        return None

    text = str(value)
    operations = [
        ("company_abbreviation", abbreviate_company),
        ("ocr_char_confusion", apply_ocr_confusion),
        ("missing_character", drop_random_char),
        ("transposed_characters", swap_adjacent_chars),
        ("case_variation", lambda text, rng: change_case(text, rng)),
    ]
    rng.shuffle(operations)
    for error_type, operation in operations:
        corrupted = operation(text, rng)
        if corrupted and corrupted != text:
            return corrupted, error_type
    return None


def corrupt_number(value: Any, rng: random.Random) -> tuple[Any, str] | None:
    if not isinstance(value, (int, float)):
        return None

    operations = ["decimal_rounding", "small_amount_delta", "digit_transposition"]
    rng.shuffle(operations)

    for error_type in operations:
        if error_type == "decimal_rounding":
            rounded = round(float(value), rng.choice([0, 1]))
            if rounded != value:
                return rounded, error_type

        if error_type == "small_amount_delta":
            delta = rng.choice([-1, 1]) * rng.uniform(0.01, 2.50)
            noisy = round(float(value) + delta, 2)
            if noisy >= 0 and noisy != value:
                return noisy, error_type

        if error_type == "digit_transposition":
            text = f"{value}"
            digit_positions = [idx for idx, char in enumerate(text) if char.isdigit()]
            adjacent = [
                idx
                for idx in digit_positions
                if idx + 1 in digit_positions and text[idx] != text[idx + 1]
            ]
            if adjacent:
                idx = rng.choice(adjacent)
                noisy_text = text[:idx] + text[idx + 1] + text[idx] + text[idx + 2 :]
                try:
                    if isinstance(value, int):
                        return int(float(noisy_text)), error_type
                    return round(float(noisy_text), 2), error_type
                except ValueError:
                    pass

    return None


def corrupt_date(value: Any, rng: random.Random) -> tuple[Any, str] | None:
    if not maybe(value):
        return None

    text = str(value)
    parsed: datetime | None = None
    for fmt in ("%d.%m.%Y", "%d/%m/%Y", "%Y-%m-%d"):
        try:
            parsed = datetime.strptime(text, fmt)
            break
        except ValueError:
            continue

    if parsed is None:
        return corrupt_text(text, rng)

    operation = rng.choice(["date_format_variation", "day_month_swap", "date_offset"])
    if operation == "date_format_variation":
        fmt = rng.choice(["%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y", "%d-%m-%y"])
        return parsed.strftime(fmt), operation

    if operation == "day_month_swap" and parsed.day <= 12:
        return f"{parsed.month:02d}.{parsed.day:02d}.{parsed.year}", operation

    offset = rng.choice([-3, -2, -1, 1, 2, 3])
    return (parsed + timedelta(days=offset)).strftime("%d.%m.%Y"), "date_offset"


def set_nested_value(document: dict[str, Any], path: tuple[Any, ...], value: Any) -> None:
    current: Any = document
    for part in path[:-1]:
        current = current[part]
    current[path[-1]] = value


def get_nested_value(document: dict[str, Any], path: tuple[Any, ...]) -> Any:
    current: Any = document
    for part in path:
        current = current[part]
    return current


def changed_path(path: tuple[Any, ...]) -> str:
    return ".".join(str(part) for part in path)


def add_changed_field(changed_fields: list[str], path: tuple[Any, ...]) -> None:
    path_text = changed_path(path)
    if path_text not in changed_fields:
        changed_fields.append(path_text)


def collect_product_descriptions(invoice: dict[str, Any]) -> list[str]:
    products = invoice.get("products")
    if not isinstance(products, list):
        return []

    descriptions = []
    for product in products:
        if isinstance(product, dict) and maybe(product.get("description")):
            descriptions.append(str(product["description"]))
    return descriptions


def build_product_description_pool(rows: list[dict[str, Any]]) -> list[str]:
    descriptions: list[str] = []
    for row in rows:
        descriptions.extend(collect_product_descriptions(row["ground_truth"]))
    return sorted(set(descriptions))


def apply_buyer_seller_swap(
    invoice: dict[str, Any],
    trace: NoiseTrace,
    changed_fields: list[str],
) -> bool:
    field_pairs = [
        (("buyer", "company_name"), ("seller", "company_name")),
        (("buyer", "name"), ("seller", "company_name")),
        (("buyer", "address"), ("seller", "address")),
    ]

    swapped_any = False
    for buyer_path, seller_path in field_pairs:
        if not safely_has_path(invoice, buyer_path) or not safely_has_path(invoice, seller_path):
            continue

        buyer_value = get_nested_value(invoice, buyer_path)
        seller_value = get_nested_value(invoice, seller_path)
        if not maybe(buyer_value) or not maybe(seller_value) or buyer_value == seller_value:
            continue

        set_nested_value(invoice, buyer_path, seller_value)
        set_nested_value(invoice, seller_path, buyer_value)
        add_changed_field(changed_fields, buyer_path)
        add_changed_field(changed_fields, seller_path)
        swapped_any = True

    if swapped_any:
        trace.add("buyer_seller_swap")
    return swapped_any


def apply_product_description_shuffle(
    invoice: dict[str, Any],
    rng: random.Random,
    trace: NoiseTrace,
    changed_fields: list[str],
) -> bool:
    products = invoice.get("products")
    if not isinstance(products, list):
        return False

    description_paths = [
        ("products", product_idx, "description")
        for product_idx, product in enumerate(products)
        if isinstance(product, dict) and maybe(product.get("description"))
    ]
    if len(description_paths) < 2:
        return False

    descriptions = [get_nested_value(invoice, path) for path in description_paths]
    shuffled = descriptions[:]
    for _attempt in range(5):
        rng.shuffle(shuffled)
        if shuffled != descriptions:
            break
    if shuffled == descriptions:
        return False

    for path, description in zip(description_paths, shuffled):
        set_nested_value(invoice, path, description)
        add_changed_field(changed_fields, path)

    trace.add("product_description_shuffle")
    return True


def apply_product_description_replace(
    invoice: dict[str, Any],
    description_pool: list[str],
    rng: random.Random,
    trace: NoiseTrace,
    changed_fields: list[str],
    level: str,
) -> bool:
    products = invoice.get("products")
    if not isinstance(products, list) or not description_pool:
        return False

    description_paths = [
        ("products", product_idx, "description")
        for product_idx, product in enumerate(products)
        if isinstance(product, dict) and maybe(product.get("description"))
    ]
    if not description_paths:
        return False

    replace_count = 1 if level != "high" else rng.randint(1, min(2, len(description_paths)))
    rng.shuffle(description_paths)
    replaced_any = False

    for path in description_paths[:replace_count]:
        current_description = str(get_nested_value(invoice, path))
        replacement_candidates = [
            description for description in description_pool if description != current_description
        ]
        if not replacement_candidates:
            continue

        set_nested_value(invoice, path, rng.choice(replacement_candidates))
        add_changed_field(changed_fields, path)
        replaced_any = True

    if replaced_any:
        trace.add("product_description_replace")
    return replaced_any


def apply_structural_noise(
    invoice: dict[str, Any],
    level: str,
    description_pool: list[str],
    rng: random.Random,
    trace: NoiseTrace,
    changed_fields: list[str],
) -> None:
    probabilities = structural_noise_probabilities(level)
    operations = [
        (
            "buyer_seller_swap",
            lambda: apply_buyer_seller_swap(invoice, trace, changed_fields),
        ),
        (
            "product_description_shuffle",
            lambda: apply_product_description_shuffle(invoice, rng, trace, changed_fields),
        ),
        (
            "product_description_replace",
            lambda: apply_product_description_replace(
                invoice, description_pool, rng, trace, changed_fields, level
            ),
        ),
    ]
    rng.shuffle(operations)

    for error_type, operation in operations:
        if rng.random() < probabilities[error_type]:
            operation()


def collect_noise_candidates(invoice: dict[str, Any]) -> list[tuple[tuple[Any, ...], str]]:
    candidates: list[tuple[tuple[Any, ...], str]] = [
        (("seller", "company_name"), "company"),
        (("seller", "address"), "text"),
        (("seller", "vat_no"), "text"),
        (("buyer", "company_name"), "company"),
        (("buyer", "name"), "text"),
        (("buyer", "address"), "text"),
        (("buyer", "country"), "text"),
        (("invoice", "number"), "text"),
        (("invoice", "date"), "date"),
        (("payment", "discount_amount"), "number"),
        (("payment", "discount_percentage"), "number"),
        (("payment", "due"), "number"),
        (("payment", "sub_total"), "number"),
        (("payment", "total"), "number"),
    ]

    products = invoice.get("products") or []
    for product_idx, product in enumerate(products[:3]):
        if isinstance(product, dict):
            candidates.extend(
                [
                    (("products", product_idx, "description"), "text"),
                    (("products", product_idx, "quantity"), "number"),
                    (("products", product_idx, "unit_price"), "number"),
                    (("products", product_idx, "total_price"), "number"),
                ]
            )
    return [
        (path, kind)
        for path, kind in candidates
        if safely_has_path(invoice, path) and maybe(get_nested_value(invoice, path))
    ]


def safely_has_path(document: dict[str, Any], path: tuple[Any, ...]) -> bool:
    current: Any = document
    for part in path:
        if isinstance(current, dict) and part in current:
            current = current[part]
        elif isinstance(current, list) and isinstance(part, int) and part < len(current):
            current = current[part]
        else:
            return False
    return True


def corrupt_value(value: Any, kind: str, rng: random.Random) -> tuple[Any, str] | None:
    if kind == "company":
        return corrupt_company(value, rng)
    if kind == "date":
        return corrupt_date(value, rng)
    if kind == "number":
        return corrupt_number(value, rng)
    return corrupt_text(value, rng)


def create_noisy_invoice(
    ground_truth: dict[str, Any],
    rng: random.Random,
    product_description_pool: list[str],
) -> tuple[dict[str, Any], NoiseTrace, list[str]]:
    noisy = deepcopy(ground_truth)
    level = choose_noise_level(rng)
    trace = NoiseTrace(level=level)
    changed_fields: list[str] = []
    apply_structural_noise(noisy, level, product_description_pool, rng, trace, changed_fields)

    candidates = collect_noise_candidates(noisy)
    rng.shuffle(candidates)

    target_operations = operations_for_level(level, rng)
    for path, kind in candidates:
        if len(changed_fields) >= target_operations:
            break

        current_value = get_nested_value(noisy, path)
        corrupted = corrupt_value(current_value, kind, rng)
        if corrupted is None:
            continue

        noisy_value, error_type = corrupted
        if noisy_value == current_value:
            continue

        set_nested_value(noisy, path, noisy_value)
        trace.add(error_type)
        add_changed_field(changed_fields, path)

    if level == "high" and rng.random() < 0.35:
        removable_paths = [
            path
            for path, _kind in candidates
            if ".".join(str(part) for part in path) not in changed_fields
        ]
        if removable_paths:
            path = rng.choice(removable_paths)
            set_nested_value(noisy, path, None)
            trace.add("missing_value")
            add_changed_field(changed_fields, path)

    return noisy, trace, changed_fields


def product_count(invoice: dict[str, Any]) -> int:
    products = invoice.get("products")
    return len(products) if isinstance(products, list) else 0


def get(invoice: dict[str, Any], *path: str, default: Any = None) -> Any:
    current: Any = invoice
    for part in path:
        if not isinstance(current, dict) or part not in current:
            return default
        current = current[part]
    return current


def flatten_record(
    record_id: int,
    source_image: str | None,
    ground_truth: dict[str, Any],
    noisy: dict[str, Any],
    trace: NoiseTrace,
    changed_fields: list[str],
) -> dict[str, Any]:
    fields = {
        "seller_company_name": ("seller", "company_name"),
        "seller_vat_no": ("seller", "vat_no"),
        "seller_address": ("seller", "address"),
        "buyer_company_name": ("buyer", "company_name"),
        "buyer_name": ("buyer", "name"),
        "buyer_country": ("buyer", "country"),
        "buyer_address": ("buyer", "address"),
        "invoice_number": ("invoice", "number"),
        "invoice_date": ("invoice", "date"),
        "payment_sub_total": ("payment", "sub_total"),
        "payment_discount_amount": ("payment", "discount_amount"),
        "payment_discount_percentage": ("payment", "discount_percentage"),
        "payment_due": ("payment", "due"),
        "payment_total": ("payment", "total"),
    }

    flat: dict[str, Any] = {
        "record_id": record_id,
        "source_image": source_image,
        "noise_level": trace.level,
        "error_types": "|".join(trace.errors),
        "changed_fields": "|".join(changed_fields),
        "product_count": product_count(ground_truth),
    }

    for name, path in fields.items():
        flat[f"gt_{name}"] = get(ground_truth, *path)
        flat[f"noisy_{name}"] = get(noisy, *path)

    flat["gt_products_json"] = json.dumps(
        ground_truth.get("products", []), ensure_ascii=False, sort_keys=True
    )
    flat["noisy_products_json"] = json.dumps(
        noisy.get("products", []), ensure_ascii=False, sort_keys=True
    )
    return flat


def write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = list(rows[0].keys())
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        for row in rows:
            file.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\n")


def primary_error_type(row: dict[str, Any]) -> str:
    errors = row.get("error_types") or []
    if isinstance(errors, list) and errors:
        return str(errors[0])
    if isinstance(errors, str) and errors:
        return errors.split("|", maxsplit=1)[0]
    return "none"


def split_key(row: dict[str, Any]) -> tuple[str, str]:
    return str(row.get("noise_level", "unknown")), primary_error_type(row)


def allocate_proportional_counts(
    group_sizes: dict[tuple[str, str], int], target_total: int
) -> dict[tuple[str, str], int]:
    total_size = sum(group_sizes.values())
    if target_total < 0 or target_total > total_size:
        raise ValueError("Target split size must be between 0 and the dataset size")

    allocations: dict[tuple[str, str], int] = {}
    remainders: list[tuple[float, tuple[str, str]]] = []
    for key, size in group_sizes.items():
        exact = size * target_total / total_size
        count = min(size, int(exact))
        allocations[key] = count
        remainders.append((exact - count, key))

    remaining = target_total - sum(allocations.values())
    remainders.sort(key=lambda item: (-item[0], item[1]))
    for _remainder, key in remainders:
        if remaining == 0:
            break
        if allocations[key] < group_sizes[key]:
            allocations[key] += 1
            remaining -= 1

    return allocations


def create_evaluation_split(
    rows: list[dict[str, Any]],
    validation_share: float,
    test_share: float,
    seed: int,
) -> tuple[set[int], set[int]]:
    if validation_share < 0 or test_share < 0 or validation_share + test_share > 1:
        raise ValueError("Validation and test shares must be non-negative and sum to 1 or less")

    rng = random.Random(seed)
    target_validation = round(len(rows) * validation_share)
    target_test = round(len(rows) * test_share)
    target_evaluation = target_validation + target_test

    groups: dict[tuple[str, str], list[dict[str, Any]]] = {}
    for row in rows:
        groups.setdefault(split_key(row), []).append(row)

    group_sizes = {key: len(group_rows) for key, group_rows in groups.items()}
    eval_counts = allocate_proportional_counts(group_sizes, target_evaluation)

    selected_by_group: dict[tuple[str, str], list[dict[str, Any]]] = {}
    for key, group_rows in groups.items():
        shuffled = group_rows[:]
        rng.shuffle(shuffled)
        selected_by_group[key] = shuffled[: eval_counts[key]]

    selected_sizes = {
        key: len(group_rows)
        for key, group_rows in selected_by_group.items()
        if group_rows
    }
    validation_counts = allocate_proportional_counts(selected_sizes, target_validation)

    validation_ids: set[int] = set()
    test_ids: set[int] = set()
    for key, selected_rows in selected_by_group.items():
        validation_count = validation_counts.get(key, 0)
        for row in selected_rows[:validation_count]:
            validation_ids.add(int(row["record_id"]))
        for row in selected_rows[validation_count:]:
            test_ids.add(int(row["record_id"]))

    if len(validation_ids) != target_validation or len(test_ids) != target_test:
        raise ValueError("Could not create evaluation splits with the requested sizes")

    return validation_ids, test_ids


def add_evaluation_split(
    rows: list[dict[str, Any]], validation_ids: set[int], test_ids: set[int]
) -> list[dict[str, Any]]:
    evaluation_rows: list[dict[str, Any]] = []
    for row in rows:
        record_id = int(row["record_id"])
        if record_id in validation_ids:
            evaluation_rows.append({"evaluation_split": "validation", **row})
        elif record_id in test_ids:
            evaluation_rows.append({"evaluation_split": "test", **row})
    return evaluation_rows




## 2. Configurazione

Modifica qui seed, split e percorsi di input/output.

In [ ]:
# Configurazione notebook
INPUT_PATH = DEFAULT_INPUT
CSV_OUTPUT_PATH = DEFAULT_CSV_OUTPUT
JSONL_OUTPUT_PATH = DEFAULT_JSONL_OUTPUT
CORPUS_OUTPUT_PATH = DEFAULT_CORPUS_OUTPUT
EVAL_CSV_OUTPUT_PATH = DEFAULT_EVAL_CSV_OUTPUT
EVAL_JSONL_OUTPUT_PATH = DEFAULT_EVAL_JSONL_OUTPUT
VALIDATION_SHARE = 0.15
TEST_SHARE = 0.15
SEED = 42

INPUT_PATH, CSV_OUTPUT_PATH, JSONL_OUTPUT_PATH, CORPUS_OUTPUT_PATH, EVAL_CSV_OUTPUT_PATH, EVAL_JSONL_OUTPUT_PATH

## 3. Generazione dataset

Questa cella riscrive i file di output principali.

In [ ]:
rng = random.Random(SEED)
source_rows = load_ground_truth(INPUT_PATH)
product_description_pool = build_product_description_pool(source_rows)

flat_rows = []
jsonl_rows = []
corpus_rows = []

for source_row in source_rows:
    ground_truth = source_row["ground_truth"]
    noisy, trace, changed_fields = create_noisy_invoice(
        ground_truth,
        rng,
        product_description_pool,
    )

    corpus_rows.append(
        {
            "record_id": source_row["record_id"],
            "source_image": source_row["source_image"],
            "ground_truth": ground_truth,
        }
    )
    flat_rows.append(
        flatten_record(
            record_id=source_row["record_id"],
            source_image=source_row["source_image"],
            ground_truth=ground_truth,
            noisy=noisy,
            trace=trace,
            changed_fields=changed_fields,
        )
    )
    jsonl_rows.append(
        {
            "record_id": source_row["record_id"],
            "source_image": source_row["source_image"],
            "noise_level": trace.level,
            "error_types": trace.errors,
            "changed_fields": changed_fields,
            "ground_truth": ground_truth,
            "noisy_invoice": noisy,
        }
    )

validation_ids, test_ids = create_evaluation_split(
    jsonl_rows,
    validation_share=VALIDATION_SHARE,
    test_share=TEST_SHARE,
    seed=SEED,
)
eval_flat_rows = add_evaluation_split(flat_rows, validation_ids, test_ids)
eval_jsonl_rows = add_evaluation_split(jsonl_rows, validation_ids, test_ids)

write_csv(CSV_OUTPUT_PATH, flat_rows)
write_jsonl(JSONL_OUTPUT_PATH, jsonl_rows)
write_jsonl(CORPUS_OUTPUT_PATH, corpus_rows)
write_csv(EVAL_CSV_OUTPUT_PATH, eval_flat_rows)
write_jsonl(EVAL_JSONL_OUTPUT_PATH, eval_jsonl_rows)

print(f"Loaded records: {len(source_rows)}")
print(f"CSV written: {CSV_OUTPUT_PATH}")
print(f"JSONL written: {JSONL_OUTPUT_PATH}")
print(f"Corpus JSONL written: {CORPUS_OUTPUT_PATH}")
print(f"Evaluation CSV written: {EVAL_CSV_OUTPUT_PATH}")
print(f"Evaluation JSONL written: {EVAL_JSONL_OUTPUT_PATH}")
print(f"Validation noisy queries: {len(validation_ids)}")
print(f"Test noisy queries: {len(test_ids)}")

## 4. Controllo rapido dei nuovi errori

In [ ]:
from collections import Counter

error_counts = Counter(error for row in jsonl_rows for error in row["error_types"])
noise_counts = Counter(row["noise_level"] for row in jsonl_rows)

print("Noise levels:", dict(noise_counts))
print("Nuovi errori:")
for error_type in ["buyer_seller_swap", "product_description_shuffle", "product_description_replace"]:
    print(f"- {error_type}: {error_counts[error_type]}")